# ABSA IndoBERT — Run Experiment (Google Colab, dengan MLflow)

Notebook ini menjalankan tahapan `run_experiment.py` (validasi data → persiapan data → training → evaluasi) **langsung memanggil fungsi pipeline** — jadi progres training dan hasil evaluasi tetap tercetak langsung di output cell — sambil mencatat parameter, metrik, dan artefak ke **MLflow**, setara dengan yang dilakukan `run_experiment()`.

**Sebelum jalan:**
- Runtime → Change runtime type → pilih **GPU** (T4 dsb).
- Siapkan MLflow tracking server yang bisa diakses **dari Colab** — Colab jalan di VM Google, jadi `http://localhost:5000` di file config **tidak bisa diakses** dari sana. Expose server lokal lewat tunnel (`ngrok http 5000`, `cloudflared tunnel`, dst.) atau pakai server yang sudah di-deploy publik, lalu isi URL-nya di variabel `MLFLOW_TRACKING_URI` pada Cell "5. Muat konfigurasi" di bawah.
- Kalau ada cell yang gagal di tengah jalan (mis. validasi data gagal), MLflow run akan tertinggal berstatus aktif — jalankan `mlflow.end_run()` manual sebelum run ulang.

## 1. (Opsional) Mount Google Drive
Hanya dipakai di akhir notebook untuk menyalin checkpoint `best_model.pt` supaya tidak hilang saat runtime Colab di-reset. Lewati cell ini kalau tidak perlu — checkpoint tetap tersimpan di storage lokal Colab (`model_output/`) selama sesi berjalan.

In [ ]:
MOUNT_DRIVE = True  # set False kalau tidak mau pakai Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 2. Clone / update repository

In [ ]:
import os

REPO_URL = "https://github.com/cupskii/Model_ABSA.git"
REPO_DIR = "/content/Model_ABSA"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

## 3. Install dependencies
Cell ini otomatis **restart runtime sekali** setelah install — perlu, karena `numpy` bawaan Colab sudah ke-*load* di proses sebelum `pip install` meng-upgrade versi di disk, sehingga modul native lama & Python layer baru tercampur (`ImportError: cannot import name '_center' from 'numpy._core.umath'`, dst). Setelah runtime restart otomatis, jalankan ulang notebook dari cell paling atas (Runtime → Run all, atau Ctrl+F9) — pada run kedua cell ini akan mendeteksi dependency sudah terpasang & tidak restart lagi.

In [ ]:
%pip install -q -r requirements.txt

import os

_RESTART_MARKER = "/content/.deps_restart_done"

if not os.path.exists(_RESTART_MARKER):
    with open(_RESTART_MARKER, "w") as f:
        f.write("1")
    print("\nDependency terpasang. Restart runtime sekali supaya numpy dkk ter-load bersih...")
    print("Setelah restart otomatis: Runtime > Run all (atau Ctrl+F9) untuk lanjut.")
    os.kill(os.getpid(), 9)
else:
    print("Dependency sudah terpasang & runtime sudah pernah di-restart — lanjut.")

## 4. Cek GPU

In [ ]:
import torch
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Tidak ada GPU — Runtime > Change runtime type > GPU, lalu jalankan ulang dari Cell 1.")

## 5. Muat konfigurasi eksperimen & mulai MLflow run
Ganti `CONFIG_PATH` kalau mau pakai config lain dari folder `configs/`. Isi `MLFLOW_TRACKING_URI` dengan URL server MLflow yang **bisa diakses dari Colab** (bukan `localhost`) — kosongkan untuk memakai `mlflow.tracking_uri` dari file config.

Server MLflow-nya jalan dengan `--no-serve-artifacts`, jadi upload artifact (requirements.txt, checkpoint, dst.) dilakukan **langsung dari Colab ke bucket S3-compatible (B2)** — perlu 4 kredensial S3 (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `MLFLOW_S3_ENDPOINT_URL`, `AWS_DEFAULT_REGION`, sama seperti isi `.env` lokal kamu). Paling aman: isi lewat **Colab Secrets** (ikon kunci 🔑 di sidebar) dengan nama persis 4 di atas + aktifkan "Notebook access". Kalau tidak ada, cell di bawah minta lewat prompt tersembunyi.

In [ ]:
import os
import yaml
import mlflow
from getpass import getpass

os.chdir("/content/Model_ABSA")  # jaga-jaga kalau lanjut habis runtime restart tanpa re-run Cell 2

from run_experiment import flatten_config, get_git_commit

# Kredensial S3 (B2) untuk upload artifact langsung dari client -- server
# MLflow jalan dengan --no-serve-artifacts, lihat docker-compose.yaml.
for _name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "MLFLOW_S3_ENDPOINT_URL", "AWS_DEFAULT_REGION"):
    if not os.environ.get(_name):
        try:
            from google.colab import userdata
            _val = userdata.get(_name)
        except Exception:
            _val = None
        os.environ[_name] = _val or getpass(f"{_name}: ")

CONFIG_PATH = "configs/experiment_indobert_baseline.yaml"
MLFLOW_TRACKING_URI = ""  # contoh: "https://xxxx.ngrok-free.app" — kosongkan utk pakai config["mlflow"]["tracking_uri"]

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f"Eksperimen : {config['experiment']['name']}")
print(f"Deskripsi  : {config['experiment'].get('description', '-')}")
print(f"Model      : {config['representation']['model_name']}")

seed = config['experiment'].get('seed', 42)
git_commit = get_git_commit()

tracking_uri = MLFLOW_TRACKING_URI or config["mlflow"]["tracking_uri"]
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(config["experiment"]["name"])

run = mlflow.start_run(run_name=config["experiment"].get("run_name", config["experiment"]["name"]))
run_id = run.info.run_id

mlflow.set_tag("git_commit", git_commit)
mlflow.set_tag("model_name", config["representation"]["model_name"])
mlflow.set_tag("model_revision", config["representation"].get("model_revision", "main"))
mlflow.set_tag("mlflow.note.content", config["experiment"].get("description", ""))
mlflow.log_param("experiment.seed", seed)
for k, v in flatten_config(config).items():
    mlflow.log_param(k, str(v)[:500])
mlflow.log_artifact("requirements.txt")

print(f"\nTracking URI : {tracking_uri}")
print(f"MLflow Run ID: {run_id}")
config

## 6. Validasi data
Langkah 1/4 — pengecekan kolom teks, label, dan nilai yang valid.

In [ ]:
from model.absa_model import set_seed
from pipeline.validate_data import validate_data

set_seed(seed)

val_report = validate_data(config)
if not val_report['passed']:
    raise RuntimeError(f"Validasi data gagal: {val_report['issues']}")

print(f"OK — {val_report['total_rows']} baris, semua pemeriksaan lulus")
if val_report['issues']:
    print(f"Peringatan: {val_report['issues']}")

mlflow.log_param('data.n_rows', val_report['total_rows'])

## 7. Persiapan data
Langkah 2/4 — preprocessing teks + split train/val/test.

In [ ]:
import json

from pipeline.prepare_data import prepare_data

data = prepare_data(config)

n_train, n_val, n_test = len(data['df_train']), len(data['df_val']), len(data['df_test'])
print(f"Train: {n_train} | Val: {n_val} | Test: {n_test}")

mlflow.log_param('data.n_train', n_train)
mlflow.log_param('data.n_val', n_val)
mlflow.log_param('data.n_test', n_test)

save_dir = config['model']['save_dir']
os.makedirs(save_dir, exist_ok=True)
cw_path = os.path.join(save_dir, 'class_weights.json')
with open(cw_path, 'w', encoding='utf-8') as f:
    json.dump(data['class_weights'], f, indent=2, ensure_ascii=False)

data['df_train'].head()

## 8. Training model
Langkah 3/4 — progres tiap epoch (train/val loss, detection F1, sentiment F1) tercetak langsung di bawah. Checkpoint terbaik otomatis tersimpan ke `config['model']['save_dir']` (default: `model_output/`).

In [ ]:
from pipeline.train_model import train_model

trained = train_model(config, data)

print(f"\nBest Val Sentiment F1 : {trained['best_val_f1']:.4f}")
print(f"Best Val Detection F1 : {trained['best_val_det_f1']:.4f}")

mlflow.log_metric('best_val_sentiment_f1', trained['best_val_f1'])
mlflow.log_metric('best_val_detection_f1', trained['best_val_det_f1'])

## 9. Evaluasi pada test set
Langkah 4/4 — laporan lengkap per aspek + classification report tercetak langsung di bawah, hasil akhir juga dikembalikan sebagai dict metrik.

In [ ]:
from pipeline.evaluate_model import evaluate_model

metrics = evaluate_model(config, trained, data)
mlflow.log_metrics(metrics)

### Classification report & confusion matrix
Teks classification report sudah tercetak di output Cell 18 di atas (`evaluate_model` memanggil `print()` langsung). Cell di bawah ini menampilkannya lagi secara rapi dari file yang tersimpan, plus confusion matrix per aspek yang aslinya cuma disimpan ke PNG dan tidak otomatis tampil di notebook (`evaluate_model()` memanggil `plt.close()` setelah menyimpan gambarnya).

In [ ]:
from IPython.display import Image, display

save_dir = trained['save_dir']

report_path = os.path.join(save_dir, 'classification_report.txt')
with open(report_path, 'r', encoding='utf-8') as f:
    print(f.read())

cm_path = os.path.join(save_dir, 'confusion_matrix.png')
display(Image(filename=cm_path))

mlflow.log_artifact(report_path, artifact_path='model_artifacts')
mlflow.log_artifact(cm_path, artifact_path='model_artifacts')

## 10. Ringkasan metrik

In [ ]:
import pandas as pd

print(f"Test Mean Sentiment F1 : {metrics.get('test_mean_sentiment_f1', 0):.4f}  <- metrik utama")
print(f"Test Mean Detection F1 : {metrics.get('test_mean_detect_f1', 0):.4f}")
print(f"Test Pair-based Micro F1     : {metrics.get('test_pair_micro_f1', 0):.4f}")
print(f"Test Aspect Detection F1     : {metrics.get('test_aspect_detection_f1', 0):.4f}")

pd.DataFrame([metrics]).T.rename(columns={0: "value"})

In [ ]:
import matplotlib.pyplot as plt

from preprocessing.preprocessing_functions import FINAL_ASPECTS
from pipeline.evaluate_model import _asp_key

sent_f1 = [metrics[f"test_{_asp_key(a)}_sentiment_f1"] for a in FINAL_ASPECTS]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(FINAL_ASPECTS, sent_f1)
ax.set_ylabel("Sentiment F1")
ax.set_title("Test Sentiment F1 per Aspek")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 11. (Opsional) Salin checkpoint ke Google Drive & tutup MLflow run
Supaya `best_model.pt`, tokenizer, dan config tidak hilang saat runtime Colab direset. Cell ini juga mencatat lokasi checkpoint sebagai tag MLflow, meng-upload config sebagai artefak, lalu menutup MLflow run.

In [ ]:
import shutil

run_name = config['experiment'].get('run_name', config['experiment']['name'])

if MOUNT_DRIVE and os.path.isdir('/content/drive/MyDrive'):
    save_dir  = config['model']['save_dir']
    dest_dir  = f"/content/drive/MyDrive/absa_models/{run_name}"

    os.makedirs(dest_dir, exist_ok=True)
    for fname in os.listdir(save_dir):
        shutil.copy2(os.path.join(save_dir, fname), dest_dir)
    print(f"Checkpoint disalin ke {dest_dir}")

    mlflow.set_tag('model_checkpoint_uri', f"gdrive://MyDrive/absa_models/{run_name}")
else:
    print("Drive tidak di-mount — checkpoint hanya tersimpan lokal di sesi Colab ini.")
    mlflow.set_tag('model_checkpoint_uri', f"local://{os.path.abspath(config['model']['save_dir'])}")

mlflow.set_tag('model_checkpoint_run_id', run_id)
mlflow.log_artifact(CONFIG_PATH, artifact_path='config')

mlflow.end_run()
print(f"\nMLflow run selesai: {run_id}")